# Ch 10. Regularization — Solutions

> This notebook contains rigorous solutions and reproducible checks for all five exercises.

## Problem 1

**Problem**: Compare MNIST MLP test accuracy for Dropout $p=0, 0.2, 0.5, 0.8$.

### Rigorous solution

Controlled factor: **dropout probability: 0, 0.2, 0.5, 0.8**.  Reported metric: **test accuracy on a fixed noisy synthetic split**.  Interpretation and limitation: Data, initialization, optimizer, and steps are matched. The offline task measures regularization under noise; it does not claim MNIST numbers.


In [ ]:
import torch
torch.manual_seed(100); X=torch.randn(400,20); y=(X[:,:4].sum(1)+.8*torch.randn(400)>0).long(); tr=slice(0,300); te=slice(300,None)
for p in (0.,.2,.5,.8):
 torch.manual_seed(101); m=torch.nn.Sequential(torch.nn.Linear(20,32),torch.nn.ReLU(),torch.nn.Dropout(p),torch.nn.Linear(32,2)); o=torch.optim.Adam(m.parameters(),lr=.03); m.train()
 for _ in range(100): loss=torch.nn.functional.cross_entropy(m(X[tr]),y[tr]); o.zero_grad(); loss.backward(); o.step()
 m.eval(); print({'dropout_p':p,'test_accuracy':(m(X[te]).argmax(1)==y[te]).float().mean().item()})


## Problem 2

**Problem**: Apply BatchNorm and LayerNorm to the same MLP and compare learning curves.

### Rigorous solution

Controlled factor: **normalization: BatchNorm versus LayerNorm**.  Reported metric: **training loss curve endpoints**.  Interpretation and limitation: The same MLP and batches are used. BatchNorm normalizes each feature across the batch; LayerNorm normalizes features within each example, so batch composition affects only the former.


In [ ]:
import torch
torch.manual_seed(102); X=torch.randn(128,12)*torch.linspace(1,4,12); y=(X[:,:3].sum(1)>0).long()
for kind in ('batch','layer'):
 torch.manual_seed(103); norm=torch.nn.BatchNorm1d(24) if kind=='batch' else torch.nn.LayerNorm(24); m=torch.nn.Sequential(torch.nn.Linear(12,24),norm,torch.nn.ReLU(),torch.nn.Linear(24,2)); o=torch.optim.SGD(m.parameters(),lr=.1); curve=[]
 for _ in range(50): loss=torch.nn.functional.cross_entropy(m(X),y); o.zero_grad(); loss.backward(); o.step(); curve.append(loss.item())
 print({'normalization':kind,'initial_loss':curve[0],'final_loss':curve[-1]})


## Problem 3

**Problem**: Compare RMSNorm and LayerNorm in a small Transformer (implemented in the next chapter).

### Rigorous solution

Controlled factor: **normalization: LayerNorm versus RMSNorm**.  Reported metric: **output mean, RMS, and runtime per repeated call**.  Interpretation and limitation: LayerNorm centers and scales; RMSNorm only scales. Both use identical input. Timing is measured locally and is descriptive, not a universal speed claim.


In [ ]:
import torch
import time
torch.manual_seed(104); x=torch.randn(64,32,128); ln=torch.nn.LayerNorm(128,elementwise_affine=False)
for name,fn in [('layernorm',lambda z:ln(z)),('rmsnorm',lambda z:z/torch.sqrt(z.square().mean(-1,keepdim=True)+1e-5))]:
 t=time.perf_counter();
 for _ in range(30): out=fn(x)
 dt=(time.perf_counter()-t)/30; print({'norm':name,'mean':out.mean().item(),'rms':out.square().mean().sqrt().item(),'seconds_per_call':dt})


## Problem 4

**Problem**: Compare learning curves for weight decay $\lambda = 0, 10^{-4}, 10^{-2}, 10^{-1}$.

### Rigorous solution

Controlled factor: **weight decay: 0, 1e-4, 1e-2, 1e-1**.  Reported metric: **test loss and weight norm**.  Interpretation and limitation: A fixed noisy regression split isolates L2 strength. Stronger decay directly penalizes norm but can underfit, so both norm and held-out loss are reported.


In [ ]:
import torch
torch.manual_seed(105); X=torch.randn(240,12); true=torch.randn(12,1); y=X@true+.8*torch.randn(240,1); tr=slice(0,180); te=slice(180,None)
for wd in (0.,1e-4,1e-2,1e-1):
 torch.manual_seed(106); m=torch.nn.Linear(12,1); o=torch.optim.SGD(m.parameters(),lr=.05,weight_decay=wd)
 for _ in range(120): loss=torch.nn.functional.mse_loss(m(X[tr]),y[tr]); o.zero_grad(); loss.backward(); o.step()
 print({'weight_decay':wd,'test_loss':torch.nn.functional.mse_loss(m(X[te]),y[te]).item(),'weight_norm':m.weight.norm().item()})


## Problem 5

**Problem**: Experiment with how gradient clipping affects training stability at a large learning rate.

### Rigorous solution

Controlled factor: **gradient clipping: off versus max norm 1**.  Reported metric: **maximum update norm and final loss at a large learning rate**.  Interpretation and limitation: Clipping rescales only gradients above the threshold. Reporting update size and loss directly tests stability on a deliberately ill-scaled regression.


In [ ]:
import torch
torch.manual_seed(107); X=torch.randn(64,4)*10; y=X.sum(1,keepdim=True)
for clip in (None,1.):
 torch.manual_seed(108); m=torch.nn.Linear(4,1); losses=[]; updates=[]
 for _ in range(20):
  loss=torch.nn.functional.mse_loss(m(X),y); m.zero_grad(); loss.backward()
  if clip is not None: torch.nn.utils.clip_grad_norm_(m.parameters(),clip)
  with torch.no_grad(): updates.append((.02*m.weight.grad).norm().item()); m.weight-=.02*m.weight.grad; m.bias-=.02*m.bias.grad
  losses.append(loss.item())
 print({'clip':clip,'max_update_norm':max(updates),'final_loss':losses[-1]})
